# Lesson 9.1: How do you build a RAG system that understands images and tables?

**Companion notebook for Lesson 9.1**

---

| Section | What you will build |
|---|---|
| 1. The Multimodal Problem | Prove that text-only RAG fails on visual questions — with real numbers |
| 2. Pipeline Anatomy | Visualize the five-stage multimodal RAG pipeline |
| 3. Approach 1: Captioning | Generate synthetic charts, caption them, index captions alongside text |
| 4. Approach 2: Multimodal Embeddings | Mock CLIP-style joint embedding space — text queries matching image vectors |
| 5. Decision Framework | Flowchart + comparison matrix — when to use which approach |
| 6. Common Mistakes | Code examples of each pitfall the blog warns about |
| 7. Evaluation | 8-query benchmark: text-only vs. Approach 1 vs. combined |
| 8. Claude API | Real vision-model captioning on the charts you generated |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Section 8):** `anthropic`

> **Document used throughout:** a synthetic *TechCo Annual Report 2023* — four visual elements  
> (bar chart, line chart, table, pie chart) whose specific values do not appear anywhere in the surrounding text.
> That gap is the multimodal RAG problem in a nutshell.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Section 8

%matplotlib inline
import os, io, base64, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from IPython.display import Image as IPImage, display

warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS']        = '1'
os.environ['MKL_NUM_THREADS']        = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')


In [ ]:
# Synthetic TechCo Annual Report 2023
# TEXT_CHUNKS = what text-only RAG indexes.
# VISUALS = charts and tables that contain the *specific* numbers.
# The text pages deliberately omit those numbers — they only describe trends in prose.

TEXT_CHUNKS = [
    {'id': 'txt_1', 'page': 1,  'title': 'Executive Summary',
     'text': 'TechCo delivered record results in 2023, driven by strong growth across all '
             'business segments. Cloud services, enterprise software, and consumer hardware '
             'all achieved double-digit year-over-year growth. We expanded our global '
             'footprint to 47 countries and continued to invest heavily in AI capabilities.'},
    {'id': 'txt_2', 'page': 3,  'title': 'Financial Highlights',
     'text': 'Fiscal year 2023 was defined by disciplined execution and revenue '
             'diversification. Operating margins improved year-over-year despite significant '
             'R&D investment. Free cash flow reached a new company record, enabling continued '
             'share buybacks and strategic acquisitions.'},
    {'id': 'txt_3', 'page': 7,  'title': 'Revenue Performance',
     'text': 'Quarterly revenue performance remained strong throughout 2023, with each quarter '
             'building on the momentum of the previous period. Seasonal patterns were consistent '
             'with historical trends, and our subscription model provided reliable recurring revenue.'},
    {'id': 'txt_4', 'page': 9,  'title': 'Stock Performance',
     'text': 'TechCo shares appreciated meaningfully in 2023, outperforming major market indices. '
             'We attribute this to investor confidence in our long-term strategy, consistent '
             'earnings beats, and successful product launches in the second half of the year.'},
    {'id': 'txt_5', 'page': 12, 'title': 'Market Position',
     'text': 'TechCo maintained and expanded its competitive position across key segments in 2023. '
             'Our investments in product quality, customer success, and go-to-market efficiency '
             'drove market share gains in several categories.'},
    {'id': 'txt_6', 'page': 15, 'title': 'Product Portfolio',
     'text': 'Our diversified product portfolio continues to be a key competitive advantage. '
             'By serving customers across multiple segments, we reduce dependency on any single '
             'revenue stream and can cross-sell effectively across our installed base.'},
]

VISUALS = [
    {'id': 'fig_1', 'page': 7,  'type': 'bar_chart',
     'title': 'Quarterly Revenue 2022-2023',
     'data': {
         'labels': ['Q1 2022','Q2 2022','Q3 2022','Q4 2022','Q1 2023','Q2 2023','Q3 2023','Q4 2023'],
         'values': [42, 48, 51, 63, 57, 68, 74, 86], 'unit': '$B',
     }},
    {'id': 'fig_2', 'page': 9,  'type': 'line_chart',
     'title': 'TechCo Stock vs S&P 500 (Jan-Dec 2023)',
     'data': {
         'months':    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
         'techco':    [  0,   4,   9,  14,  22,  29,  38,  42,  44,  47,  53,  58],
         'sp500':     [  0,   3,   5,   8,  10,  13,  15,  17,  16,  18,  21,  24],
     }},
    {'id': 'tbl_1', 'page': 12, 'type': 'table',
     'title': 'Market Share by Segment (2023)',
     'data': {
         'headers': ['Segment', 'TechCo Share', 'Nearest Competitor', 'YoY Change'],
         'rows': [
             ['Cloud Infrastructure', '31%', '42%', '+3pp'],
             ['Enterprise SaaS',      '18%', '24%', '+2pp'],
             ['Consumer Devices',     '22%', '35%', '+1pp'],
             ['Security Software',    '14%', '19%', '+4pp'],
         ],
     }},
    {'id': 'fig_3', 'page': 15, 'type': 'pie_chart',
     'title': 'Revenue by Product Line (FY 2023)',
     'data': {
         'labels': ['Cloud Services','Enterprise Software','Consumer Hardware','Professional Services'],
         'values': [38, 29, 21, 12], 'unit': '%',
     }},
]

print(f'Document: {len(TEXT_CHUNKS)} text pages + {len(VISUALS)} visual elements')
print()
print('Visual elements (these contain data NOT present in any text page):')
for v in VISUALS:
    print(f'  [{v["id"]}] p{v["page"]} — {v["title"]} ({v["type"]})')


In [ ]:
def make_chart(visual):
    """Render a visual element as a matplotlib figure, return as base64 PNG."""
    d, t = visual['data'], visual['type']
    fig, ax = plt.subplots(figsize=(8, 4))

    if t == 'bar_chart':
        x = np.arange(len(d['labels']))
        clrs = ['#90CAF9'] * 4 + ['#1565C0'] * 4
        bars = ax.bar(x, d['values'], color=clrs, alpha=0.9, width=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels(d['labels'], rotation=20, ha='right')
        ax.set_ylabel(f"Revenue ({d['unit']})")
        ax.axvline(3.5, color='gray', linestyle='--', alpha=0.5, label='2022 | 2023')
        ax.legend()
        for bar, val in zip(bars, d['values']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f'${val}B', ha='center', va='bottom', fontsize=9)

    elif t == 'line_chart':
        months = d['months']
        x = np.arange(len(months))
        ax.plot(x, d['techco'], 'b-o', label='TechCo', linewidth=2, markersize=5)
        ax.plot(x, d['sp500'],  'r--s', label='S&P 500', linewidth=2, markersize=5)
        ax.axhline(0, color='gray', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(months)
        ax.set_ylabel('Return (%)')
        ax.legend()
        ax.annotate(f"+{d['techco'][-1]}%", xy=(len(months)-1, d['techco'][-1]),
                    xytext=(-2.5, 4), textcoords='offset points',
                    fontsize=11, color='blue', fontweight='bold')
        ax.annotate(f"+{d['sp500'][-1]}%", xy=(len(months)-1, d['sp500'][-1]),
                    xytext=(-2.5, -14), textcoords='offset points',
                    fontsize=11, color='red', fontweight='bold')

    elif t == 'pie_chart':
        colors = ['#1565C0', '#2E7D32', '#F57F17', '#6A1B9A']
        wedges, texts, autotexts = ax.pie(
            d['values'], labels=d['labels'], colors=colors,
            autopct='%1.0f%%', startangle=90, pctdistance=0.78)
        for at in autotexts:
            at.set_fontsize(11)
            at.set_fontweight('bold')
        ax.axis('equal')

    elif t == 'table':
        ax.axis('off')
        tbl = ax.table(
            cellText=d['rows'], colLabels=d['headers'],
            cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
        tbl.auto_set_font_size(True)
        tbl.set_fontsize(10)
        for j in range(len(d['headers'])):
            tbl[0, j].set_facecolor('#1565C0')
            tbl[0, j].set_text_props(color='white', fontweight='bold')
        for i in range(1, len(d['rows']) + 1):
            for j in range(len(d['headers'])):
                tbl[i, j].set_facecolor('#F5F5F5' if i % 2 == 0 else 'white')

    ax.set_title(visual['title'], fontweight='bold', pad=12)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=96, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()


print('Generating visuals...')
for v in VISUALS:
    v['b64'] = make_chart(v)
    print(f'  [{v["id"]}] {v["title"]:40s}  {len(v["b64"]):6d} bytes (base64 PNG)')
print('Done. Displaying...')


In [ ]:
# Display all four visual elements as they would appear in the PDF
for v in VISUALS:
    print(f'  [{v["id"]}] p{v["page"]} — {v["title"]} ({v["type"]})')
    display(IPImage(data=base64.b64decode(v['b64']), format='png', width=650))
    print()


---
## 1. The Multimodal Problem — What Text RAG Misses

The annual report has text pages AND visual elements. A user asking about specific numbers
from the charts will get no useful answer from text-only RAG — the numbers simply aren't
in any text chunk.

```
User: "What was Q3 2023 revenue?"
Text-only RAG: retrieves 'Revenue Performance' page — only says 'remained strong'
                          — cannot say $74B because that number is only in the bar chart
```


In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

text_embeds = embedder.encode(
    [c['text'] for c in TEXT_CHUNKS],
    convert_to_tensor=True, show_progress_bar=False)

def text_only_retrieve(query, top_k=2):
    q_emb  = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, text_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(TEXT_CHUNKS[i], float(scores[i])) for i in idx]


VISUAL_QUESTIONS = [
    {'query': 'What was Q3 2023 revenue?',
     'correct': '$74B',              'source': 'fig_1 (bar chart)'},
    {'query': 'How did TechCo stock compare to the S&P 500 in 2023?',
     'correct': 'TechCo +58%, S&P 500 +24%', 'source': 'fig_2 (line chart)'},
    {'query': 'What is TechCo cloud infrastructure market share?',
     'correct': '31% (+3pp YoY)',    'source': 'tbl_1 (table)'},
    {'query': 'Which product line generates the most revenue?',
     'correct': 'Cloud Services 38%', 'source': 'fig_3 (pie chart)'},
]

print(f'Embedder ready. Corpus: {len(TEXT_CHUNKS)} text pages (no visuals indexed).\n')
print('=== Text-Only RAG on Visual Questions ===\n')

for item in VISUAL_QUESTIONS:
    results = text_only_retrieve(item['query'])
    top, score = results[0]
    print(f'Q: {item["query"]}')
    print(f'   Best match : [{top["id"]}] p{top["page"]} "{top["title"]}" (score={score:.3f})')
    print(f'   Text says  : "{top["text"][:70]}..."')
    print(f'   Correct ans: {item["correct"]}  (lives in {item["source"]})')
    print(f'   Text-RAG   : CANNOT ANSWER — specific values not in any text page')
    print()


---
## 2. Anatomy of the Multimodal RAG Pipeline

Five stages — each one has to handle every modality:

```
Query Understanding → Retrieval → Fusion → Augmentation → Generation
```

The key difference from standard RAG is that the **knowledge base** is now mixed-modality
(text chunks + image embeddings + table representations), and the **generator** is a
vision-language model that can look at the original image when answering.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.set_xlim(0, 15)
ax.set_ylim(0, 4)
ax.axis('off')

stages = [
    (1.5, 'Query\nUnderstanding', '#1A237E',
     'Encode text\n+ image\n+ audio query'),
    (4.5, 'Retrieval', '#1565C0',
     'Search mixed-\nmodality index\n(text+images+video)'),
    (7.5, 'Fusion', '#0288D1',
     'Merge signals\nacross modalities\n(cross-attention)'),
    (10.5, 'Augmentation', '#00838F',
     'Re-rank, filter\nenrich context\niterative retrieval'),
    (13.5, 'Generation', '#2E7D32',
     'Vision-LLM\ngenerates grounded\nmultimodal answer'),
]

for x, label, color, detail in stages:
    rect = mpatches.FancyBboxPatch((x - 1.2, 1.2), 2.4, 1.6,
                                   boxstyle='round,pad=0.15',
                                   facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 2.3, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    ax.text(x, 0.85, detail, ha='center', va='top',
            fontsize=8, color='#444444', multialignment='center')

for i in range(len(stages) - 1):
    x1 = stages[i][0] + 1.2
    x2 = stages[i+1][0] - 1.2
    ax.annotate('', xy=(x2, 2.0), xytext=(x1, 2.0),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=2.5))

ax.text(7.5, 3.85, 'Multimodal RAG Pipeline',
        ha='center', va='top', fontsize=14, fontweight='bold', color='#1A237E')

show_plot()


---
## 3. Approach 1: Captioning Pipeline — Text-ify Everything at Index Time

**Recipe:**
1. Extract images and tables from your document
2. Caption each visual with a vision model
3. Store the caption as a text chunk (alongside a pointer to the original image)
4. Embed and index — same text pipeline you already have
5. At generation time, optionally re-pass the original image to a vision LLM

The ceiling on answer quality is **caption quality**. A generic caption
(`"a bar chart showing some data"`) won't help retrieval.
A specific caption (`"Q3 2023 revenue was $74B..."`) will.


In [ ]:
class MockCaptioner:
    """
    Simulates a vision model generating captions for charts and tables.
    Uses known metadata to produce accurate, retrieval-optimised descriptions.

    Replace with a real call in Section 8 (Claude API).
    Prompt to use: 'Describe this image in detail. What kind of chart/table is it?
    List every number, axis label, trend, and annotation you can see.'
    """

    def caption(self, visual):
        d, t, title = visual['data'], visual['type'], visual['title']

        if t == 'bar_chart':
            vals  = d['values']
            labs  = d['labels']
            peak  = labs[int(np.argmax(vals))]
            q3_23 = vals[6]  # index 6 = Q3 2023
            yoy   = vals[7] - vals[3]
            return (
                f"Bar chart titled '{title}'. Shows quarterly revenue in $B for eight quarters. "
                f"Values: {', '.join(f'{l}: ${v}B' for l, v in zip(labs, vals))}. "
                f"Q3 2023 revenue was ${q3_23}B. "
                f"Peak quarter: {peak} at ${max(vals)}B. "
                f"Q4 2023 increased ${yoy}B year-over-year versus Q4 2022."
            )

        elif t == 'line_chart':
            tc_end = d['techco'][-1]
            sp_end = d['sp500'][-1]
            tc_q3  = d['techco'][8]   # September
            return (
                f"Line chart titled '{title}'. Two lines plotted across 12 months. "
                f"TechCo (blue): started 0%, ended +{tc_end}% in December. "
                f"S&P 500 (red dashed): started 0%, ended +{sp_end}% in December. "
                f"TechCo outperformed S&P 500 by {tc_end - sp_end} percentage points. "
                f"TechCo showed strongest acceleration in Q3, reaching +{tc_q3}% by September."
            )

        elif t == 'table':
            rows  = d['rows']
            hdrs  = d['headers']
            best  = max(rows, key=lambda r: r[3])
            items = '; '.join(
                f"{r[0]}: {r[1]} share (competitor: {r[2]}, YoY: {r[3]})" for r in rows)
            return (
                f"Table titled '{title}' with columns: {', '.join(hdrs)}. "
                f"Data: {items}. "
                f"Fastest growing segment: {best[0]} at {best[3]} year-over-year."
            )

        elif t == 'pie_chart':
            labs  = d['labels']
            values = d['values']
            top   = labs[int(np.argmax(values))]
            pairs = ', '.join(f'{l}: {v}%' for l, v in zip(labs, values))
            combined = values[0] + values[1]
            return (
                f"Pie chart titled '{title}'. "
                f"Segments: {pairs}. "
                f"Largest segment: {top} at {max(values)}%. "
                f"Cloud Services and Enterprise Software together account for {combined}% of revenue."
            )

        return f"Visual element: {title}"


captioner = MockCaptioner()

print('=== Captioning all visuals ===\n')
for v in VISUALS:
    caption = captioner.caption(v)
    v['caption'] = caption
    print(f'[{v["id"]}] {v["title"]}')
    print(f'Caption: {caption}')
    print()


In [ ]:
# Build the multimodal index: text chunks + captions stored as text
# Each caption chunk also carries a pointer back to the original image.

MM_CHUNKS = []

for chunk in TEXT_CHUNKS:
    MM_CHUNKS.append({
        'id': chunk['id'], 'text': chunk['text'],
        'source_type': 'text', 'page': chunk['page'], 'title': chunk['title'],
    })

for v in VISUALS:
    MM_CHUNKS.append({
        'id': v['id'], 'text': v['caption'],
        'source_type': v['type'], 'page': v['page'], 'title': v['title'],
        'has_image': True, 'b64': v['b64'],   # pointer to original
    })

mm_embeds = embedder.encode(
    [c['text'] for c in MM_CHUNKS],
    convert_to_tensor=True, show_progress_bar=False)

def mm_retrieve(query, top_k=2):
    q_emb  = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, mm_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(MM_CHUNKS[i], float(scores[i])) for i in idx]


print(f'Multimodal index: {len(MM_CHUNKS)} chunks  '
      f'({len(TEXT_CHUNKS)} text + {len(VISUALS)} caption chunks)')
print()
print('=== Approach 1 (Captioning) on the same visual questions ===\n')

for item in VISUAL_QUESTIONS:
    results   = mm_retrieve(item['query'])
    top, score = results[0]
    is_visual  = top.get('has_image', False)
    icon       = 'CHART/TABLE' if is_visual else 'TEXT'
    hit        = item['correct'].split()[0].lower() in top['text'].lower()
    mark       = 'YES' if hit else 'PARTIAL'
    print(f'Q: {item["query"]}')
    print(f'   Top match  : [{icon}] [{top["id"]}] "{top["title"]}" (score={score:.3f})')
    print(f'   Contains "{item["correct"]}": {mark}')
    if is_visual:
        print(f'   Caption snip: {top["text"][:100]}...')
    print()


---
## 4. Approach 2: Multimodal Embeddings — One Vector Space for Text and Images

Instead of converting images to text, use a model like **CLIP** that maps both modalities
into the same vector space. Now:

```
text_embed("quarterly revenue chart")  ≈  image_embed(📊 bar chart image)
```

A text query can **directly match** an image vector — no captioning step needed at index time.

At generation time, you pass the actual image to a vision-LLM (Claude, GPT-4o, Gemini)
which reads the chart values directly.

**Production options:** CLIP, BLIP-2, ColPali (page-as-image), Vertex Multimodal Embeddings.

The demo below mocks CLIP by giving each image a semantic descriptor and embedding that —
illustrating *why* the joint space works, without the full model weight download.


In [ ]:
# Mock CLIP: each image is represented by a semantic descriptor.
# In a real CLIP model the IMAGE PIXELS produce an embedding in the same space
# as the TEXT. We simulate that by embedding a description of the image content.

IMAGE_DESCRIPTORS = {
    'fig_1': 'quarterly revenue bar chart 2022 2023 financial results year-over-year growth',
    'fig_2': 'stock performance line chart TechCo S&P 500 comparison annual return 2023',
    'tbl_1': 'market share table segments cloud enterprise consumer security competitors percentage',
    'fig_3': 'revenue breakdown pie chart product lines cloud software hardware services proportion',
}

img_ids    = list(IMAGE_DESCRIPTORS.keys())
img_embeds = embedder.encode(
    list(IMAGE_DESCRIPTORS.values()),
    convert_to_tensor=True, show_progress_bar=False)

def clip_search(query_text, top_k=2):
    q_emb  = embedder.encode(query_text, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, img_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(img_ids[i], float(scores[i]), IMAGE_DESCRIPTORS[img_ids[i]]) for i in idx]


print('=== Approach 2: text query → image vector matching (mock CLIP) ===\n')

clip_demos = [
    ('What was Q3 2023 revenue?',
     'Should match fig_1 — the revenue bar chart'),
    ('How did TechCo stock compare to S&P 500?',
     'Should match fig_2 — the line chart'),
    ('Which segment has the highest market share growth?',
     'Should match tbl_1 — the market share table'),
    ('Which product generates the biggest slice of revenue?',
     'Should match fig_3 — the pie chart'),
]

for q, note in clip_demos:
    results = clip_search(q, top_k=2)
    print(f'Query : {q}')
    print(f'Expect: {note}')
    for img_id, score, desc in results:
        print(f'  [{img_id}] score={score:.3f}  {desc[:60]}')
    print()


In [ ]:
# Visualise the shared embedding space (2D PCA projection).
# Images and semantically related text queries land close together.
# That is exactly how CLIP retrieval works.

def pca_2d(X):
    """Simple 2D PCA — avoids the sklearn import."""
    Xc = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T


TEXT_QUERIES = {
    'Q: Q3 2023 revenue':       'What was Q3 2023 revenue?',
    'Q: stock vs S&P 500':      'How did TechCo stock compare to S&P 500?',
    'Q: market share':          'What is TechCo cloud market share?',
    'Q: product breakdown':     'Which product line generates the most revenue?',
}

all_labels, all_texts, all_colors, all_markers = [], [], [], []

for img_id, desc in IMAGE_DESCRIPTORS.items():
    all_labels.append(f'IMG: {img_id}')
    all_texts.append(desc)
    all_colors.append('#1565C0')
    all_markers.append('s')

for label, text in TEXT_QUERIES.items():
    all_labels.append(label)
    all_texts.append(text)
    all_colors.append('#E53935')
    all_markers.append('^')

for chunk in TEXT_CHUNKS[:4]:
    all_labels.append(f'TXT: {chunk["id"]}')
    all_texts.append(chunk['text'])
    all_colors.append('#888888')
    all_markers.append('o')

raw_embeds = embedder.encode(all_texts, show_progress_bar=False)
coords     = pca_2d(raw_embeds)

fig, ax = plt.subplots(figsize=(13, 7))
for (x, y), label, color, marker in zip(coords, all_labels, all_colors, all_markers):
    ax.scatter(x, y, c=color, marker=marker, s=160, alpha=0.88, zorder=3, edgecolors='white', linewidth=0.8)
    ax.annotate(label, (x, y), textcoords='offset points', xytext=(8, 4), fontsize=8.5, color=color)

legend_handles = [
    mpatches.Patch(color='#1565C0', label='Image vectors (Approach 2 / CLIP)'),
    mpatches.Patch(color='#E53935', label='Query vectors'),
    mpatches.Patch(color='#888888', label='Text chunk vectors'),
]
ax.legend(handles=legend_handles, loc='lower left', fontsize=10)
ax.set_title(
    'Shared Embedding Space (PCA 2D)\n'
    'A query vector near an image vector = the text query matches that image',
    fontweight='bold')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
show_plot()


---
## 5. Decision Framework — When to Use Which Approach

The blog's rule of thumb:

> If a smart intern with a notepad could read your documents and write down
> everything that matters in plain English, **Approach 1** is enough.
> If they'd need to literally show you the picture, you need **Approach 2**.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
ax.set_xlim(0, 13)
ax.set_ylim(0, 14)
ax.axis('off')

def fbox(ax, x, y, w, h, text, fc='#1565C0', tc='white', fs=10):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                 boxstyle='round,pad=0.12', facecolor=fc, edgecolor='#BBBBBB', lw=1.5))
    ax.text(x, y, text, ha='center', va='center', fontsize=fs,
            color=tc, fontweight='bold', multialignment='center')

def farrow(ax, x1, y1, x2, y2, label='', lpos='left'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333333', lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        offset = (-0.4, 0) if lpos == 'left' else (0.4, 0)
        ax.text(mx + offset[0], my + offset[1], label, fontsize=9, color='#555555',
                ha='center', va='center')

# Start node
fbox(ax, 6.5, 13.2, 7, 0.8, 'New question arrives', fc='#37474F')
farrow(ax, 6.5, 12.8, 6.5, 12.1)

# Q1
fbox(ax, 6.5, 11.65, 8, 0.85,
     'Can the answer be written out\nin plain text without losing meaning?',
     fc='#F57F17', tc='black')
farrow(ax, 6.5, 11.22, 6.5, 10.5, 'YES')
farrow(ax, 10.5, 11.65, 11.5, 11.65, 'NO', lpos='right')

# Approach 1 box
fbox(ax, 6.5, 10.05, 5.5, 0.75, 'Approach 1: Captioning\n(text-ify at index time)', fc='#2E7D32')
farrow(ax, 6.5, 9.67, 6.5, 8.9)

# Q2
fbox(ax, 6.5, 8.5, 8, 0.75,
     'Do users query with images, or\ndoes visual similarity matter?',
     fc='#F57F17', tc='black')
farrow(ax, 6.5, 8.12, 6.5, 7.4, 'NO')
farrow(ax, 10.5, 8.5, 11.5, 8.5, 'YES', lpos='right')

# Approach 1 enough
fbox(ax, 6.5, 7.0, 5.5, 0.7,
     'Approach 1 is sufficient\nAdd captioning to existing pipeline', fc='#2E7D32')
farrow(ax, 6.5, 6.65, 6.5, 5.9)

# Q3
fbox(ax, 6.5, 5.5, 8, 0.75,
     'Is spatial layout or visual structure\nthe core meaning (MRI, circuit, map)?',
     fc='#F57F17', tc='black')
farrow(ax, 6.5, 5.12, 6.5, 4.4, 'NO')
farrow(ax, 10.5, 5.5, 11.5, 5.5, 'YES', lpos='right')

# Pragmatic
fbox(ax, 6.5, 4.0, 5.5, 0.7,
     'Pragmatic: Approach 1 + 2\nCaption AND multimodal embed', fc='#1565C0')

# Approach 2 (right branch)
fbox(ax, 11.5, 10, 2.8, 2.5,
     'Approach 2\nMultimodal\nEmbeddings\n(CLIP / ColPali)', fc='#E53935', fs=9)
ax.annotate('', xy=(11.5, 8.9), xytext=(11.5, 11.22),
            arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.8))
ax.annotate('', xy=(11.5, 6.3), xytext=(11.5, 4.75),
            arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.8))

ax.set_title('Decision Flowchart: Which Multimodal RAG Approach?',
             fontsize=14, fontweight='bold', y=0.98)
show_plot()


---
## 6. Common Mistakes

The blog lists five. Here is each one as runnable code — showing the bad pattern
and the fix side by side.


In [ ]:
# ── Mistake 1: No pointer back to the original image ─────────────────────────
print('=== Mistake 1: Losing the pointer to the source image ===\n')

bad_entry  = {'id': 'fig_1', 'text': 'Bar chart: Q3 2023 revenue $74B...'}
good_entry = {'id': 'fig_1', 'text': 'Bar chart: Q3 2023 revenue $74B...',
              'source_file': 'annual_report_2023.pdf',
              'source_page': 7, 'source_type': 'bar_chart',
              'b64': VISUALS[0]['b64'][:20] + '...(PNG bytes)'}  # pointer

print('BAD  — caption stored alone:')
print(f'  {bad_entry}')
print()
print('GOOD — caption + pointer to source:')
for k, v in good_entry.items():
    print(f'  {k}: {str(v)[:70]}')

print()
print('Why it matters:')
print('  Without the pointer: you cannot show the user the original chart.')
print('  Without the pointer: you cannot pass the image to a vision LLM at generation time.')
print('  Without the pointer: you cannot regenerate captions when a better model ships.')

print()
print('-' * 60)
# ── Mistake 2: Tables serialized without headers ───────────────────────────────
print()
print('=== Mistake 2: Serializing tables without column headers ===\n')

tbl = VISUALS[2]['data']
bad_serial  = ' '.join(' '.join(r) for r in tbl['rows'])
good_serial = '\n'.join(
    '| ' + ' | '.join(tbl['headers']) + ' |' if i == 0
    else '| ' + ' | '.join(r) + ' |'
    for i, r in enumerate([tbl['headers']] + tbl['rows']))

print(f'BAD  (raw flatten): "{bad_serial[:80]}"')
print()
print('GOOD (markdown table):')
print(good_serial)
print()
print('A query for "cloud infrastructure market share" will match the GOOD version;')
print('31% 42% +3pp in isolation is meaningless without the column headers.')

print()
print('-' * 60)
# ── Mistake 3: Generic captions ───────────────────────────────────────────────
print()
print('=== Mistake 3: Generic captions that do not help retrieval ===\n')

bad_caption  = 'A bar chart showing some financial data for a company over time.'
good_caption = VISUALS[0]['caption']

query = 'What was Q3 2023 revenue?'
q_emb = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
s_bad  = float(util.cos_sim(q_emb, embedder.encode(bad_caption,  convert_to_tensor=True, show_progress_bar=False)))
s_good = float(util.cos_sim(q_emb, embedder.encode(good_caption, convert_to_tensor=True, show_progress_bar=False)))

print(f'Query: "{query}"')
print()
print(f'Generic caption  (score={s_bad:.3f}): "{bad_caption}"')
print(f'Specific caption (score={s_good:.3f}): "{good_caption[:100]}..."')
print()
print(f'Retrieval score improvement: +{(s_good - s_bad)*100:.0f}% from specific values in caption.')
print('Lesson: force the vision model to list every number it can see.')


---
## 7. Evaluation — Text-Only vs. Approach 1 (Captioning) vs. Combined

8 queries: 5 require visual data, 3 are text-only. Scoring: 1.0 = answer found in
retrieved context, 0.0 = not found.

This operationalises the blog's *"Try it yourself"* exercise:
> "Run those questions through your current text-only RAG. How many does it get right?
> That number is your multimodal RAG starting point."


In [ ]:
EVAL_SET = [
    # Visual questions — specific numbers only in charts/tables
    {'query': 'What was Q3 2023 revenue?',
     'key': '$74',         'source': 'fig_1', 'type': 'visual',
     'label': 'Q3 2023 revenue (bar chart)'},
    {'query': 'How did TechCo stock compare to the S&P 500 in 2023?',
     'key': '58',          'source': 'fig_2', 'type': 'visual',
     'label': 'Stock vs S&P 500 (line chart)'},
    {'query': 'What is TechCo cloud infrastructure market share?',
     'key': '31%',         'source': 'tbl_1', 'type': 'visual',
     'label': 'Cloud market share (table)'},
    {'query': 'Which product line generates the most revenue?',
     'key': 'Cloud Services', 'source': 'fig_3', 'type': 'visual',
     'label': 'Top product line (pie chart)'},
    {'query': 'What is the fastest growing market segment by share?',
     'key': 'Security',    'source': 'tbl_1', 'type': 'visual',
     'label': 'Fastest growing segment (table)'},
    # Text questions — both approaches should handle these
    {'query': 'How many countries does TechCo operate in?',
     'key': '47',          'source': 'txt_1', 'type': 'text',
     'label': 'Country count (text)'},
    {'query': 'What drives TechCo competitive advantage?',
     'key': 'portfolio',   'source': 'txt_6', 'type': 'text',
     'label': 'Competitive advantage (text)'},
    {'query': 'How did operating margins trend in 2023?',
     'key': 'improved',    'source': 'txt_2', 'type': 'text',
     'label': 'Operating margins (text)'},
]


def retrieve_context(query, chunks, embeds, top_k=2):
    q_emb  = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return ' '.join(chunks[i]['text'] for i in idx)


txt_scores, mm_scores = [], []

print(f'{"Label":<36} {"Type":<8} {"Text-Only":<12} {"Multimodal (A1)"}')
print('-' * 72)

for item in EVAL_SET:
    ctx_txt = retrieve_context(item['query'], TEXT_CHUNKS, text_embeds)
    ctx_mm  = retrieve_context(item['query'], MM_CHUNKS,   mm_embeds)

    t = 1.0 if item['key'].lower() in ctx_txt.lower() else 0.0
    m = 1.0 if item['key'].lower() in ctx_mm.lower()  else 0.0

    txt_scores.append(t)
    mm_scores.append(m)

    tm = 'YES' if t else 'NO '
    mm = 'YES' if m else 'NO '
    print(f'{item["label"]:<36} {item["type"]:<8} {tm:<12} {mm}')

print()
print(f'Text-only accuracy  : {np.mean(txt_scores):.0%}  ({int(sum(txt_scores))}/{len(txt_scores)})')
print(f'Multimodal accuracy : {np.mean(mm_scores):.0%}  ({int(sum(mm_scores))}/{len(mm_scores)})')
print(f'Improvement         : +{(np.mean(mm_scores) - np.mean(txt_scores)):.0%}')
print()
print('On visual questions only:')
v_idx = [i for i,e in enumerate(EVAL_SET) if e['type'] == 'visual']
print(f'  Text-only  : {np.mean([txt_scores[i] for i in v_idx]):.0%}')
print(f'  Multimodal : {np.mean([mm_scores[i]  for i in v_idx]):.0%}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-query
labels = [e['label'][:30] for e in EVAL_SET]
x      = np.arange(len(EVAL_SET))
w      = 0.38

bt = axes[0].bar(x - w/2, txt_scores, w, color='#E53935', alpha=0.85, label='Text-Only RAG')
bm = axes[0].bar(x + w/2, mm_scores,  w, color='#2E7D32', alpha=0.85, label='Multimodal RAG (Approach 1)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
axes[0].set_ylim(-0.05, 1.25)
axes[0].set_ylabel('Answer found in context (1=yes, 0=no)')
axes[0].set_title('Per-Query: Text-Only vs. Multimodal RAG', fontweight='bold')
axes[0].legend(fontsize=9)

# Shade visual vs text query zones
n_visual = sum(1 for e in EVAL_SET if e['type'] == 'visual')
axes[0].axvspan(-0.5, n_visual - 0.5, alpha=0.05, color='red')
axes[0].axvspan(n_visual - 0.5, len(EVAL_SET) - 0.5, alpha=0.05, color='green')
axes[0].text(n_visual / 2 - 0.5, 1.18, 'Visual questions', ha='center', fontsize=8, color='#C62828')
axes[0].text(n_visual + (len(EVAL_SET) - n_visual) / 2 - 0.5, 1.18,
             'Text questions', ha='center', fontsize=8, color='#1B5E20')

# Right: overall comparison
avgs   = [np.mean(txt_scores), np.mean(mm_scores)]
clrs   = ['#E53935', '#2E7D32']
slbls  = ['Text-Only\nRAG', 'Multimodal\nRAG (A1)']
bars   = axes[1].bar(slbls, avgs, color=clrs, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel('Average accuracy')
axes[1].set_title('Overall Accuracy', fontweight='bold')
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.0%}', ha='center', fontsize=14, fontweight='bold')

show_plot()

print('Takeaway: text-only RAG answers all the text questions but fails on visual questions.')
print('Captioning (Approach 1) recovers those failures at the cost of one extra index-time step.')


---
## 8. Real Vision Captioning with Claude API

Replace `MockCaptioner` with a real vision model call. The charts generated in Section 3
are already stored as base64 PNGs — pass them directly to Claude.

**Prompt design matters more than model choice.**  
Generic prompt → `"A bar chart showing financial data."` — useless for retrieval.  
Specific prompt → `"List every number, axis label, trend, and annotation visible."` — useful.

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`


In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

CAPTION_PROMPT = (
    'You are captioning a chart or table from a business document for use in a '
    'retrieval-augmented generation (RAG) system. The caption must be detailed enough '
    'that a text search for a specific data point (e.g. a revenue figure, a percentage, '
    'a product name) will match this caption.\n\n'
    'Describe:\n'
    '1. What type of visual this is (bar chart, line chart, pie chart, table, etc.)\n'
    '2. The title and axis labels (if present)\n'
    '3. Every specific number, percentage, and label you can see\n'
    '4. The main trend, comparison, or key takeaway\n'
    '5. Any notable outliers or annotations\n\n'
    'Be precise and factual. Aim for 3-5 sentences. Do not guess values you cannot read clearly.'
)

if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_caption(visual):
        resp = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=400,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image',
                     'source': {'type': 'base64', 'media_type': 'image/png',
                                'data': visual['b64']}},
                    {'type': 'text', 'text': CAPTION_PROMPT},
                ]
            }]
        )
        return resp.content[0].text.strip()

    print('=== Claude Vision Captions ===\n')
    for v in VISUALS:
        print(f'[{v["id"]}] {v["title"]}')
        claude_cap = claude_caption(v)
        print(f'Claude : {claude_cap}')
        print(f'Mock   : {v["caption"][:120]}...')
        print()

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('Caption prompt used (CAPTION_PROMPT):')
    print(CAPTION_PROMPT)
    print()
    print('Model recommendations:')
    print('  Index-time captioning at scale  : claude-haiku-4-5-20251001 (fast, cheap)')
    print('  Complex diagrams / medical scans: claude-sonnet-4-6 (better spatial reasoning)')
    print()
    print('One-sentence trick: end every caption prompt with')
    print('  "List every number you can see in this image."')
    print('It forces the model to be specific instead of describing trends in prose.')


---
## Key Takeaways

1. **Text-only RAG fails silently on visual content.** It retrieves the right *page* but the
   right *answer* is locked inside a chart or table. The score looks fine; the answer is wrong.

2. **Approach 1 (captioning) is the right starting point.** It adds one step at index time,
   reuses your existing text retriever, and recovers all visual questions whose content
   can be written in plain English. Most business documents fall into this category.

3. **Caption quality is the ceiling.** A generic caption (`"a bar chart showing some data"`) scores
   near zero for retrieval. A specific caption that lists every number scores much higher.
   Use `"List every number you can see in this image."` as a forcing function.

4. **Always store a pointer back to the original image.** The caption is for retrieval;
   the original image is for generation. Without the pointer, you cannot pass the chart
   to a vision LLM at answer time.

5. **Approach 2 (multimodal embeddings) earns its complexity for three use cases:**
   image-based queries, visual similarity search, and content where spatial layout IS the
   meaning (MRI scans, circuit diagrams, satellite imagery).

6. **Most production systems end up doing both.** Caption at index time (Approach 1) for
   text-search recall. Also store the multimodal embedding (Approach 2) for image-query
   matching. Keep the original image file. Pass it to the vision LLM at generation time.

7. **The evaluation number is your starting point.** Run your current RAG on questions
   whose answers live in visuals. Count how many it gets right. That gap is what
   multimodal RAG closes — and it is almost always larger than teams expect.

---

*Up next — Lesson 9.2: "My captions sound generic. How do I get vision models to caption
charts and tables in a way that is actually useful for retrieval?"
We will engineer the captioning prompt end-to-end and measure the retrieval lift.*
